In [0]:

# ============================================================
# Notebook 5: Model Training
# ATP Tennis Match Winner Prediction
# ============================================================

storage_account = "tennisdatalake60105845"
storage_key = "LjZoIlf5yGSIHxM/9GIXHx5jTq8VNvMOqWjuLZ54krBy3gn+BN7pg8q5/4UM8dgtAEwIMaTNyIYl+AStqzCGpQ=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Load feature set from curated zone
df = spark.read.format("delta").table("amazon_dbx_60105845.default.atp_tennis_features")

print(f"Loaded {df.count()} records")
df.show(5)

Loaded 67433 records
+---------+--------+-------------------+----------------+-------------+-------------+-------------+------------+------+------+----+-----+-----------------+
|rank_diff|pts_diff|          odds_diff|p1_higher_ranked|is_grand_slam|is_hard_court|is_clay_court|is_best_of_5|Rank_1|Rank_2|year|month|winner_is_player1|
+---------+--------+-------------------+----------------+-------------+-------------+-------------+------------+------+------+----+-----+-----------------+
|       24|    -130| 0.8400000000000001|               0|            0|            1|            0|           0|    89|    65|2013|   12|                0|
|       -5|    8775|               -0.8|               1|            1|            1|            0|           1|     1|     6|2014|    1|                1|
|      -54|     264|               -1.0|               1|            0|            0|            1|           0|    58|   112|2014|    2|                1|
|      121|   -1190|               0.97|   

In [0]:
# ============================================================
# Prepare Data for ML - Train/Test Split
# ============================================================
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Define feature columns
feature_cols = [
    "rank_diff", "pts_diff", "odds_diff", "p1_higher_ranked",
    "is_grand_slam", "is_hard_court", "is_clay_court", "is_best_of_5",
    "Rank_1", "Rank_2", "year", "month"
]

# Assemble features into a vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df).select("features", "winner_is_player1")
df_ml = df_ml.withColumnRenamed("winner_is_player1", "label")

# Train/Test split (80/20) with seed for reproducibility
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Training records: {train_df.count()}")
print(f"Testing records: {test_df.count()}")
print(f" Features: {feature_cols}")

Training records: 53810
Testing records: 13623
 Features: ['rank_diff', 'pts_diff', 'odds_diff', 'p1_higher_ranked', 'is_grand_slam', 'is_hard_court', 'is_clay_court', 'is_best_of_5', 'Rank_1', 'Rank_2', 'year', 'month']


In [0]:
# ============================================================
# Train Models - Baseline + Advanced
# ============================================================
import mlflow
import mlflow.spark

mlflow.set_experiment("/Tennis-Match-Prediction")

# ---- Model 1: Logistic Regression (Baseline) ----
with mlflow.start_run(run_name="LogisticRegression_baseline"):
    lr = LogisticRegression(maxIter=10, regParam=0.01)
    lr_model = lr.fit(train_df)
    lr_preds = lr_model.transform(test_df)
    
    evaluator_auc = BinaryClassificationEvaluator(metricName="areaUnderROC")
    evaluator_acc = MulticlassClassificationEvaluator(metricName="accuracy")
    
    lr_auc = evaluator_auc.evaluate(lr_preds)
    lr_acc = evaluator_acc.evaluate(lr_preds)
    
    mlflow.log_metric("AUC", lr_auc)
    mlflow.log_metric("Accuracy", lr_acc)
    mlflow.log_param("model", "LogisticRegression")
    
    print(f" Logistic Regression - AUC: {lr_auc:.4f} | Accuracy: {lr_acc:.4f}")

# ---- Model 2: Random Forest ----
with mlflow.start_run(run_name="RandomForest"):
    rf = RandomForestClassifier(numTrees=100, maxDepth=5, seed=42)
    rf_model = rf.fit(train_df)
    rf_preds = rf_model.transform(test_df)
    
    rf_auc = evaluator_auc.evaluate(rf_preds)
    rf_acc = evaluator_acc.evaluate(rf_preds)
    
    mlflow.log_metric("AUC", rf_auc)
    mlflow.log_metric("Accuracy", rf_acc)
    mlflow.log_param("model", "RandomForest")
    
    print(f" Random Forest - AUC: {rf_auc:.4f} | Accuracy: {rf_acc:.4f}")

# ---- Model 3: Gradient Boosted Trees ----
with mlflow.start_run(run_name="GBT"):
    gbt = GBTClassifier(maxIter=50, maxDepth=5, seed=42)
    gbt_model = gbt.fit(train_df)
    gbt_preds = gbt_model.transform(test_df)
    
    gbt_auc = evaluator_auc.evaluate(gbt_preds)
    gbt_acc = evaluator_acc.evaluate(gbt_preds)
    
    mlflow.log_metric("AUC", gbt_auc)
    mlflow.log_metric("Accuracy", gbt_acc)
    mlflow.log_param("model", "GBT")
    
    print(f" GBT - AUC: {gbt_auc:.4f} | Accuracy: {gbt_acc:.4f}")

 Logistic Regression - AUC: 0.7581 | Accuracy: 0.6871
 Random Forest - AUC: 0.7551 | Accuracy: 0.6907
 GBT - AUC: 0.7619 | Accuracy: 0.6913


In [0]:
# ============================================================
# Register Best Model (GBT) in MLflow
# ============================================================

with mlflow.start_run(run_name="GBT_final_registered"):
    gbt_final = GBTClassifier(maxIter=50, maxDepth=5, seed=42)
    gbt_final_model = gbt_final.fit(train_df)
    
    # Log parameters
    mlflow.log_param("model_type", "GradientBoostedTrees")
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("seed", 42)
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_param("test_size", test_df.count())
    mlflow.log_param("features", str(feature_cols))
    mlflow.log_param("data_version", "v1.0")
    
    # Log metrics
    gbt_final_preds = gbt_final_model.transform(test_df)
    final_auc = evaluator_auc.evaluate(gbt_final_preds)
    final_acc = evaluator_acc.evaluate(gbt_final_preds)
    
    mlflow.log_metric("AUC", final_auc)
    mlflow.log_metric("Accuracy", final_acc)
    
    # Save model to storage
    gbt_final_model.write().overwrite().save(
        f"abfss://curated@{storage_account}.dfs.core.windows.net/models/gbt_tennis_v1"
    )
    
    print(f"Best model registered!")
    print(f" AUC: {final_auc:.4f} | Accuracy: {final_acc:.4f}")
    print(f" Model saved to curated zone!")

Best model registered!
 AUC: 0.7619 | Accuracy: 0.6913
 Model saved to curated zone!


In [0]:
# ============================================================
# Model Validation & Error Analysis
# ============================================================
from pyspark.sql.functions import col, when
import pandas as pd
import matplotlib.pyplot as plt

# Predictions dataframe
preds = gbt_final_preds.select("label", "prediction", "probability")

# Confusion matrix
tp = preds.filter((col("label") == 1) & (col("prediction") == 1)).count()
tn = preds.filter((col("label") == 0) & (col("prediction") == 0)).count()
fp = preds.filter((col("label") == 0) & (col("prediction") == 1)).count()
fn = preds.filter((col("label") == 1) & (col("prediction") == 0)).count()

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)

print("=== CONFUSION MATRIX ===")
print(f"True Positives:  {tp}")
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"\n=== METRICS ===")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC:       {final_auc:.4f}")
print(f"Accuracy:  {final_acc:.4f}")

print(f"\n=== MODEL COMPARISON ===")
print(f"Logistic Regression - AUC: {lr_auc:.4f} | Accuracy: {lr_acc:.4f}")
print(f"Random Forest       - AUC: {rf_auc:.4f} | Accuracy: {rf_acc:.4f}")
print(f"GBT (Best)          - AUC: {gbt_auc:.4f} | Accuracy: {gbt_acc:.4f}")

=== CONFUSION MATRIX ===
True Positives:  4509
True Negatives:  4908
False Positives: 1971
False Negatives: 2235

=== METRICS ===
Precision: 0.6958
Recall:    0.6686
F1 Score:  0.6819
AUC:       0.7619
Accuracy:  0.6913

=== MODEL COMPARISON ===
Logistic Regression - AUC: 0.7581 | Accuracy: 0.6871
Random Forest       - AUC: 0.7551 | Accuracy: 0.6907
GBT (Best)          - AUC: 0.7619 | Accuracy: 0.6913


In [0]:
# ============================================================
# Deployment Validation - Sanity Checks
# ============================================================
from pyspark.ml.linalg import Vectors

# Test with known scenarios
print("=== DEPLOYMENT VALIDATION ===")
print("Testing model with known scenarios...\n")

# Create test cases
test_cases = spark.createDataFrame([
    # (rank_diff, pts_diff, odds_diff, p1_higher_ranked, is_grand_slam, 
    #  is_hard_court, is_clay_court, is_best_of_5, Rank_1, Rank_2, year, month)
    # Case 1: Top player (rank 1) vs weak player (rank 100) - P1 should win
    (-99, 10000, -2.0, 1, 1, 1, 0, 1, 1, 100, 2023, 6),
    # Case 2: Weak player (rank 200) vs top player (rank 5) - P2 should win  
    (195, -8000, 3.0, 0, 0, 1, 0, 0, 200, 5, 2023, 3),
    # Case 3: Equal players
    (2, 100, 0.1, 1, 0, 0, 1, 0, 50, 52, 2022, 5),
], ["rank_diff", "pts_diff", "odds_diff", "p1_higher_ranked", "is_grand_slam",
    "is_hard_court", "is_clay_court", "is_best_of_5", "Rank_1", "Rank_2", "year", "month"])

test_assembled = assembler.transform(test_cases)
test_result = gbt_final_model.transform(test_assembled)

print("Case 1 - Rank 1 vs Rank 100 (P1 should win):")
test_result.select("prediction").show(1)

print("Case 2 - Rank 200 vs Rank 5 (P2 should win):")
test_result.select("prediction").collect()[1]
print(f"  Prediction: {test_result.select('prediction').collect()[1][0]} (0=P2 wins )")

print("Case 3 - Equal players (rank 50 vs 52):")
print(f"  Prediction: {test_result.select('prediction').collect()[2][0]}")

print("\n Deployment validation complete!")
print("Model behaves as expected on sanity checks!")

=== DEPLOYMENT VALIDATION ===
Testing model with known scenarios...

Case 1 - Rank 1 vs Rank 100 (P1 should win):
+----------+
|prediction|
+----------+
|       1.0|
+----------+
only showing top 1 row
Case 2 - Rank 200 vs Rank 5 (P2 should win):
  Prediction: 0.0 (0=P2 wins )
Case 3 - Equal players (rank 50 vs 52):
  Prediction: 0.0

 Deployment validation complete!
Model behaves as expected on sanity checks!
